# Measured evaluation
Garak quick is a one-probe security smoke test, not certification. Metrics are recorded in MLflow. Inspect per-benchmark results; this release may report an inconsistent aggregate pass flag. HTML artifact export requires separate validation.


In [ ]:
%pip install "mlflow[kubernetes]==3.14.0" "boto3==1.43.18"


In [ ]:
from pathlib import Path
import sys, os
repo=Path.cwd()
if repo.name == "notebooks": repo=repo.parent
sys.path.insert(0,str(repo/"scripts"))
from science import eval_request, eval_submit, eval_export
# Nonsecret endpoints/model ID and the service CA are injected by the workbench manifest.
# The evaluation references a Secret by name; no API key is copied into this notebook.
providers=eval_request("/api/v1/evaluations/providers")
providers


In [ ]:
# This submits a real one-probe Garak evaluation. It may find a vulnerability.
job=eval_submit()
job_id=job["resource"]["id"]
print(job_id)


In [ ]:
# Re-run this cell until the job completes; inspect each benchmark's own pass flag.
result=eval_request("/api/v1/evaluations/jobs/"+job_id)
print(result["status"], result.get("results"))
if result.get("status",{}).get("state") == "completed":
    eval_export(job_id)
